# EDA and Preprocessing
## Telco Customer Churn Prediction

Going through the data before building any models.
Understanding what's in the dataset and spotting any issues early saves a lot of time later.

Sections:
1. Load and inspect the data
2. Target variable — is the data balanced?
3. Numerical features
4. Categorical features
5. Feature interactions
6. Summary and key findings

In [ ]:
import sys
import os
sys.path.append(os.path.abspath(".."))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns

sns.set_theme(style="whitegrid", palette="Set2")
plt.rcParams.update({"figure.dpi": 120, "axes.titlesize": 13, "axes.labelsize": 11,
                     "xtick.labelsize": 10, "ytick.labelsize": 10})

from src.preprocessing import load_data, clean_data, identify_column_types, split_data

## 1. Load and Inspect the Data

In [ ]:
RAW_PATH = "../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv"

df_raw = load_data(RAW_PATH)

print(f"Shape: {df_raw.shape}  ({df_raw.shape[0]} customers, {df_raw.shape[1]} columns)")
print(f"Duplicates: {df_raw.duplicated().sum()}")
df_raw.head()

In [ ]:
display(df_raw.describe())
display(df_raw.describe(include="object"))

# check which columns have missing values
null_counts = df_raw.isnull().sum()
print("Columns with null values:")
print(null_counts[null_counts > 0])

In [ ]:
df = clean_data(df_raw)
col_types = identify_column_types(df)

print("Column types after cleaning:")
for k, v in col_types.items():
    print(f"  {k}: {v}")

## 2. Target Variable — Is the Data Balanced?

This is the first thing to check. If one class is much larger than the other,
accuracy becomes a useless metric. A model that always predicts "No Churn" would
get ~73% accuracy without learning anything useful. We need F1, Precision, Recall, and AUC instead.

In [ ]:
churn_counts = df["Churn"].value_counts()
churn_pct    = df["Churn"].value_counts(normalize=True) * 100

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

bars = axes[0].bar(["No Churn (0)", "Churn (1)"], churn_counts.values,
                   color=sns.color_palette("Set2")[:2], edgecolor="white", width=0.5)
for bar, pct in zip(bars, churn_pct.values):
    axes[0].text(bar.get_x() + bar.get_width() / 2,
                 bar.get_height() + 50, f"{pct:.1f}%",
                 ha="center", va="bottom", fontweight="bold", fontsize=12)
axes[0].set_title("Churn Distribution", fontweight="bold")
axes[0].set_ylabel("Number of Customers")
axes[0].set_ylim(0, churn_counts.max() * 1.15)

axes[1].pie(churn_counts.values, labels=["No Churn", "Churn"],
            autopct="%1.1f%%", colors=sns.color_palette("Set2")[:2],
            startangle=90, explode=(0, 0.05), textprops={"fontsize": 12})
axes[1].set_title("Churn Proportion", fontweight="bold")

plt.suptitle("Target Variable — Class Imbalance", fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig("../reports/fig_01_target_distribution.png", bbox_inches="tight")
plt.show()

print(f"No Churn: {churn_counts[0]:,} ({churn_pct[0]:.1f}%)")
print(f"Churn:    {churn_counts[1]:,} ({churn_pct[1]:.1f}%)")
print(f"Imbalance ratio: {churn_counts[0]/churn_counts[1]:.1f}:1")

The dataset is quite imbalanced — roughly 73% No Churn vs 27% Churn.
This means we can't just look at accuracy. We'll use F1-score and ROC-AUC as the main metrics.

## 3. Numerical Features

Three numerical features: `tenure`, `MonthlyCharges`, `TotalCharges`.

In [ ]:
num_cols = ["tenure", "MonthlyCharges", "TotalCharges"]
palette  = {0: sns.color_palette("Set2")[0], 1: sns.color_palette("Set2")[1]}
labels   = {0: "No Churn", 1: "Churn"}

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, col in zip(axes, num_cols):
    for churn_val, color in palette.items():
        subset = df[df["Churn"] == churn_val][col]
        ax.hist(subset, bins=30, alpha=0.6, color=color,
                label=labels[churn_val], edgecolor="white")
    ax.set_title(f"{col} distribution", fontweight="bold")
    ax.set_xlabel(col)
    ax.set_ylabel("Count")
    ax.legend()

plt.suptitle("Numerical Features by Churn", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("../reports/fig_02_numerical_histograms.png", bbox_inches="tight")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, col in zip(axes, num_cols):
    sns.boxplot(data=df, x="Churn", y=col, ax=ax,
                palette="Set2", hue="Churn", legend=False)
    ax.set_title(f"{col} by Churn", fontweight="bold")
    ax.set_xlabel("Churn (0=No, 1=Yes)")
    ax.set_ylabel(col)

plt.suptitle("Box Plots — Numerical Features by Churn", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("../reports/fig_03_numerical_boxplots.png", bbox_inches="tight")
plt.show()

churn_means = df.groupby("Churn")[num_cols].mean()
display(churn_means.rename(index={0: "No Churn", 1: "Churn"}))

Churners tend to have higher monthly charges but lower total charges.
That makes sense — they're on expensive plans but leave before they've been customers for long.

In [ ]:
corr_cols = num_cols + ["Churn"]
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.heatmap(df[corr_cols].corr(), annot=True, fmt=".2f",
            cmap="RdYlGn", center=0, ax=axes[0],
            linewidths=0.5, square=True)
axes[0].set_title("Correlation Matrix", fontweight="bold")

scatter_data = df.sample(min(2000, len(df)), random_state=42)
for churn_val, color in palette.items():
    mask = scatter_data["Churn"] == churn_val
    axes[1].scatter(scatter_data.loc[mask, "tenure"],
                    scatter_data.loc[mask, "MonthlyCharges"],
                    alpha=0.4, s=15, color=color, label=labels[churn_val])
axes[1].set_xlabel("Tenure (months)")
axes[1].set_ylabel("Monthly Charges ($)")
axes[1].set_title("Tenure vs Monthly Charges", fontweight="bold")
axes[1].legend()

plt.tight_layout()
plt.savefig("../reports/fig_04_correlation_scatter.png", bbox_inches="tight")
plt.show()

## 4. Categorical Features

For each categorical feature, we compute the churn rate per category.

In [ ]:
def plot_churn_rate_by_cat(df, col, ax):
    rates = (df.groupby(col)["Churn"].mean() * 100).sort_values(ascending=True)
    colors = sns.color_palette("Set2", len(rates))
    bars = ax.barh(rates.index, rates.values, color=colors, edgecolor="white")
    for bar, val in zip(bars, rates.values):
        ax.text(val + 0.5, bar.get_y() + bar.get_height() / 2,
                f"{val:.1f}%", va="center", fontsize=9)
    ax.set_xlabel("Churn Rate (%)")
    ax.set_title(f"Churn Rate by {col}", fontweight="bold")
    ax.set_xlim(0, rates.max() * 1.2)
    return rates

highlight_cols = ["Contract", "InternetService", "PaymentMethod",
                  "OnlineSecurity", "TechSupport", "OnlineBackup"]

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for ax, col in zip(axes, highlight_cols):
    plot_churn_rate_by_cat(df, col, ax)

plt.suptitle("Churn Rate by Categorical Features", fontsize=15, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig("../reports/fig_05_categorical_churn_rates.png", bbox_inches="tight")
plt.show()

A few things stand out here:
- Month-to-month contracts have around 42% churn rate compared to about 3% for two-year contracts. Contract type looks like the strongest predictor.
- Fiber optic internet has about 42% churn vs 19% for DSL. Could be a service quality issue.
- Electronic check payment method has the highest churn rate (~45%) vs other methods.
- Customers without security, tech support, or backup services churn more.

## 5. Feature Interactions

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# churn rate broken down by contract type and internet service
pivot1 = df.groupby(["Contract", "InternetService"])["Churn"].mean().unstack() * 100
pivot1.plot(kind="bar", ax=axes[0], colormap="Set2", edgecolor="white", rot=15)
axes[0].set_title("Churn Rate: Contract x Internet Service", fontweight="bold")
axes[0].set_xlabel("Contract Type")
axes[0].set_ylabel("Churn Rate (%)")
axes[0].legend(title="Internet Service", fontsize=8)
axes[0].yaxis.set_major_formatter(mtick.PercentFormatter())

# churn rate by how long they've been a customer
df["tenure_bucket"] = pd.cut(df["tenure"], bins=[0, 12, 24, 48, 72],
                              labels=["0-12 mo", "13-24 mo", "25-48 mo", "49-72 mo"],
                              include_lowest=True)
tenure_churn = df.groupby("tenure_bucket", observed=True)["Churn"].mean() * 100
bars = axes[1].bar(tenure_churn.index, tenure_churn.values,
                   color=sns.color_palette("Set2", 4), edgecolor="white")
for bar, val in zip(bars, tenure_churn.values):
    axes[1].text(bar.get_x() + bar.get_width() / 2,
                 bar.get_height() + 0.5, f"{val:.1f}%",
                 ha="center", va="bottom", fontweight="bold")
axes[1].set_title("Churn Rate by Tenure Bucket", fontweight="bold")
axes[1].set_xlabel("Tenure Group")
axes[1].set_ylabel("Churn Rate (%)")
axes[1].yaxis.set_major_formatter(mtick.PercentFormatter())

# monthly charges distribution for churners vs non-churners
for churn_val, color in palette.items():
    subset = df[df["Churn"] == churn_val]["MonthlyCharges"]
    subset.plot.kde(ax=axes[2], color=color, label=labels[churn_val], linewidth=2)
axes[2].set_xlabel("Monthly Charges ($)")
axes[2].set_ylabel("Density")
axes[2].set_title("Monthly Charges Distribution", fontweight="bold")
axes[2].legend()
axes[2].fill_between(axes[2].lines[0].get_xdata(),
                     axes[2].lines[0].get_ydata(), alpha=0.1,
                     color=list(palette.values())[0])
axes[2].fill_between(axes[2].lines[1].get_xdata(),
                     axes[2].lines[1].get_ydata(), alpha=0.1,
                     color=list(palette.values())[1])

plt.tight_layout()
plt.savefig("../reports/fig_06_feature_interactions.png", bbox_inches="tight")
plt.show()

df.drop("tenure_bucket", axis=1, inplace=True)

## 6. Summary

In [ ]:
fig = plt.figure(figsize=(18, 12))
fig.suptitle("EDA Summary — Telco Churn", fontsize=16, fontweight="bold", y=1.01)

ax1 = fig.add_subplot(2, 3, 1)
churn_counts.plot.bar(ax=ax1, color=sns.color_palette("Set2")[:2], edgecolor="white", rot=0)
ax1.set_title("1. Class Distribution", fontweight="bold")
ax1.set_xlabel("")
ax1.set_ylabel("Count")
for bar, pct in zip(ax1.patches, churn_pct.values):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 30,
             f"{pct:.1f}%", ha="center", fontweight="bold")

ax2 = fig.add_subplot(2, 3, 2)
plot_churn_rate_by_cat(df, "Contract", ax2)

ax3 = fig.add_subplot(2, 3, 3)
plot_churn_rate_by_cat(df, "InternetService", ax3)

ax4 = fig.add_subplot(2, 3, 4)
for churn_val, color in palette.items():
    df[df["Churn"] == churn_val]["tenure"].plot.hist(
        bins=25, alpha=0.6, color=color, ax=ax4, label=labels[churn_val])
ax4.set_title("4. Tenure Distribution", fontweight="bold")
ax4.set_xlabel("Tenure (months)")
ax4.set_ylabel("Count")
ax4.legend()

ax5 = fig.add_subplot(2, 3, 5)
plot_churn_rate_by_cat(df, "TechSupport", ax5)

ax6 = fig.add_subplot(2, 3, 6)
sns.boxplot(data=df, x="Churn", y="MonthlyCharges", palette="Set2",
            hue="Churn", legend=False, ax=ax6)
ax6.set_title("6. Monthly Charges vs Churn", fontweight="bold")
ax6.set_xlabel("Churn (0=No, 1=Yes)")

plt.tight_layout()
plt.savefig("../reports/fig_07_summary_dashboard.png", bbox_inches="tight", dpi=150)
plt.show()

In [ ]:
X_train, X_test, y_train, y_test = split_data(df)

train_df = X_train.copy()
train_df["Churn"] = y_train.values
test_df = X_test.copy()
test_df["Churn"] = y_test.values

os.makedirs("../data/processed", exist_ok=True)
train_df.to_csv("../data/processed/train_raw.csv", index=False)
test_df.to_csv("../data/processed/test_raw.csv", index=False)
print("Saved train/test splits to data/processed/")

## Key Findings

| # | Finding | Impact on modelling |
|---|---------|-------------------|
| 1 | 26.5% churn rate — imbalanced dataset | Use F1 and AUC, not accuracy |
| 2 | Month-to-month contracts have ~42% churn | Will likely be the top feature |
| 3 | Higher monthly charges correlate with churn | Scale before feeding to model |
| 4 | Short tenure (< 12 months) = highest risk | Good candidate for a new feature |
| 5 | Fiber optic customers churn more than DSL | Possible service quality issue |
| 6 | No security/support/backup = higher churn | Bundle features could help |

Next: build 10 new features from these insights in `02_feature_engineering.ipynb`